# Reliability testing on dataset

In [ ]:
cd "/home/simon/Documents/Zwischen Wörtern und Pixeln/Sample_Dataset/Reichel/"

In [ ]:
#import custom Config
from Config import cfg

## Get dataset

In [ ]:
import pandas as pd

#read in dataset
df_dataset = pd.read_csv("posts_act.csv",
                        
                         sep = ";") #specify separator, otherwise the parser will cry

#verify
df_dataset.head()

## Get sample

In [ ]:
#get 200 random cases
df_sample = df_dataset.sample(n = 200, #n
                      random_state = cfg.random_seed #seed for reproducibility
                     )

#verify
df_sample.info()

## Fix filename for correct absolute path

We can use `Series.replace()` or `str.replace()`. <br>
We are going to use `str.replace()` as it is more efficient and faster (note: does not really matter for a small dataset like this).

In [ ]:
#fix filename
df_sample["filename"] = df_sample["filename"].str.replace(
    "/Reichel/media/",
    "/home/simon/Documents/Zwischen Wörtern und Pixeln/Sample_Dataset/Reichel/media_act/",
    regex = False
)

#verify
df_sample.head()

## Copy images

In [ ]:
#copy all images into a separate folder
import shutil
from pathlib import Path

#set destination
destination = Path("reliability")
destination.mkdir(parents = True, exist_ok = True)

#copy files
for file in df_sample["filename"]:
    shutil.copy2(file, destination)

## Save to csv

In [ ]:
df_sample.to_csv("reliability.csv",
                encoding = "UTF-8",
                index = False
                )

<br><br><br><br><br><br>

## Import coded and classifier file for reliability test

coded file = sample manually coded by human coder <br>
classifier file = sample coded by ML model

In [ ]:
#read data
df_classifier = pd.read_csv("classifier_results_fp32.csv")

df_human = pd.read_csv("reliability_coded.csv")

## Preprocessing

In [ ]:
#change column so they match
df_classifier = df_classifier.rename(columns = {"image_path": "filename"})

In [ ]:
#change filename so they match
df_human["filename"] = df_human["filename"].str.replace(
    "media_act",
    "reliability",
    regex = False
)

In [ ]:
#merge dataframes
df_combined = pd.merge(df_classifier,
                       df_human,
                       on = "filename"
                      )

In [ ]:
df_combined.head()

In [ ]:
#drop unnecessary columns
df_combined = df_combined.drop(["link.x", "text", "created_time", "post_type", 
                                "language_text.iso_lang_1", "platform_name", 
                                "likes", "comments", "user_name", "handle", "media_text", 
                                "confidence", "Optimistic", "Pessimistic", "Hostile", "Neutral"
                               ], axis = 1)

In [ ]:
#recode to integers
df_combined = df_combined.replace(
    ["Optimistic",
    "Pessimistic",
    "Hostile",
    "Neutral"],
    [1,
    2,
    3,
    4]
)

In [ ]:
#recode Unknowns to Zero
df_combined = df_combined.replace(
    ["Unknown"],
    [0]
)

#remove empty rows should they exist
df_combined = df_combined[
    df_combined["prediction"].notna() &
    (df_combined["prediction"] != 0) &
    (df_combined["prediction"].astype(str).str.strip() != "")
]

#remove empty rows should they exist
df_combined = df_combined[
    df_combined["code_human"].notna() &
    (df_combined["code_human"] != 0) &
    (df_combined["code_human"].astype(str).str.strip() != "")
]


In [ ]:
df_combined["prediction"] = df_combined["prediction"].astype(int)
df_combined["code_human"] = df_combined["code_human"].astype(int)

In [ ]:
print(len(df_combined))

In [ ]:
df_combined.head()

## Calculate Krippendorff's Alpha

In [ ]:
import krippendorff as kd
import numpy as np

#arrange data
reliability_data = np.array([
    df_combined["prediction"].to_numpy(),
    df_combined["code_human"].to_numpy()
])

alpha = kd.alpha(
    reliability_data = reliability_data,
    level_of_measurement = "nominal"
)

print(f"Krippendorff's alpha: {alpha:.5f}")